# Distance-Guided Meta-Learning for Molecular Activity Prediction

This notebook reproduces THEMAP's meta-learning workflow:

1. **Distances** between chemical datasets are pre-computed and saved (here, `output/molecule_distances.csv`).
2. For a chosen **target** dataset, select its **k-nearest source** datasets from that distance file.
3. **Meta-train** a Prototypical Network and a from-scratch MAML learner on those source datasets.
4. Evaluate the **low-data gain**: adapt each meta-learned model to *N* ∈ {16, 32, 64, 128} target
   samples and measure **AUROC** on the held-out remainder, versus an MLP trained from scratch on the
   same *N* samples.

The hypothesis: when sources are *close* to the target, meta-learning should beat a from-scratch
baseline most in the lowest-data regime.

In [ ]:
import os
import sys

# Run from the repository root (mirrors the other notebooks in this folder).
repo_path = os.path.dirname(os.path.abspath(""))
os.chdir(repo_path)
sys.path.insert(0, repo_path)

import matplotlib.pyplot as plt  # noqa: E402
import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402

from themap.metalearning import select_k_nearest_sources  # noqa: E402
from themap.metalearning.config import ExperimentConfig, MAMLConfig, TrainConfig  # noqa: E402
from themap.metalearning.evaluation import LowDataEvaluator  # noqa: E402
from themap.metalearning.runner import MetaLearnExperiment  # noqa: E402
from themap.utils import set_plot_style  # noqa: E402

set_plot_style()
plt.rcParams["figure.figsize"] = (10, 5)

## Configuration

In [ ]:
DATA_DIR = "datasets/"
DISTANCE_FILE = "output/molecule_distances.csv"
TARGET_ID = "CHEMBL1963831"  # a target present in the distance file (test fold)
K = 3  # number of nearest source datasets to meta-train on
FEATURIZER = "ecfp"
SUPPORT_SIZES = [16, 32, 64, 128]
SEEDS = 3  # repeated stratified draws per support size

# Modest meta-training budget so the notebook runs in a few minutes on CPU.
# Scale NUM_EPOCHS / EPISODES_PER_EPOCH up for stronger meta-learners.
TRAIN = dict(
    num_epochs=20,
    episodes_per_epoch=40,
    meta_batch_size=8,
    n_support=8,
    n_query=10,
    val_episodes=30,
    patience=8,
    device="auto",
)

## Step 1 — Select the k-nearest source datasets

We read the saved distance file and pick the `K` sources with the smallest distance to the target.

In [ ]:
selected = select_k_nearest_sources(DISTANCE_FILE, TARGET_ID, k=K)
selected_df = pd.DataFrame(selected, columns=["source_id", "distance"])
print(f"Nearest {K} sources to target {TARGET_ID}:")
display(selected_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(selected_df["source_id"], selected_df["distance"], color="#66c2a5")
ax.invert_yaxis()
ax.set_xlabel("Distance to target")
ax.set_title(f"k-nearest source datasets for {TARGET_ID}")
plt.tight_layout()
plt.show()

## Step 2 & 3 — Meta-train and evaluate ProtoNet and MAML

`MetaLearnExperiment.run()` performs the full pipeline for one algorithm: select sources, featurize
once, meta-train, then sweep the low-data support sizes on the target (meta-adapted vs from-scratch
baseline). We run it for both algorithms.

In [ ]:
def run_experiment(algorithm):
    config = ExperimentConfig(
        data_dir=DATA_DIR,
        distance_file=DISTANCE_FILE,
        target_id=TARGET_ID,
        k=K,
        algorithm=algorithm,
        featurizer=FEATURIZER,
        support_sizes=SUPPORT_SIZES,
        seeds=SEEDS,
        output_dir=f"metalearn_out/{algorithm}",
        maml=MAMLConfig(inner_lr=0.01, inner_steps=5, eval_inner_steps=16),
        train=TrainConfig(**TRAIN),
    )
    return MetaLearnExperiment(config).run()


proto_results = run_experiment("proto")
maml_results = run_experiment("maml")
proto_results.head()

## Step 4 — Compare meta-learning against the from-scratch baseline

In [ ]:
def agg(df, method):
    sub = df[df["method"] == method]
    g = sub.groupby("support_size")["auroc"]
    mean = g.mean()
    # 95% CI half-width (t-interval).
    from scipy import stats

    ci = g.apply(lambda x: stats.sem(x) * stats.t.ppf(0.975, len(x) - 1) if len(x) > 1 else np.nan)
    return mean, ci


series = {
    "ProtoNet (meta)": agg(proto_results, "meta"),
    "MAML (meta)": agg(maml_results, "meta"),
    "Baseline (from scratch)": agg(proto_results, "baseline"),
}

fig, ax = plt.subplots(figsize=(9, 5))
for label, (mean, ci) in series.items():
    ax.plot(mean.index, mean.values, marker="o", label=label)
    ax.fill_between(mean.index, mean.values - ci.values, mean.values + ci.values, alpha=0.15)
ax.set_xscale("log", base=2)
ax.set_xticks(SUPPORT_SIZES)
ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
ax.set_xlabel("Target support set size (N)")
ax.set_ylabel("AUROC on held-out target")
ax.set_title(f"Low-data meta-learning on {TARGET_ID} (k={K} nearest sources)")
ax.legend()
plt.tight_layout()
plt.show()

### Meta-learning gain over the baseline (positive = meta-learning helps)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
base_mean = series["Baseline (from scratch)"][0]
for label in ["ProtoNet (meta)", "MAML (meta)"]:
    mean = series[label][0]
    ax.plot(mean.index, (mean - base_mean).values, marker="s", label=label)
ax.axhline(0.0, color="gray", linestyle="--", linewidth=1)
ax.set_xscale("log", base=2)
ax.set_xticks(SUPPORT_SIZES)
ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
ax.set_xlabel("Target support set size (N)")
ax.set_ylabel("AUROC gain vs baseline")
ax.set_title("Does meta-learning help, and where most?")
ax.legend()
plt.tight_layout()
plt.show()

# Tabular summary.
summary = pd.concat(
    [
        LowDataEvaluator.summarize(proto_results),
        LowDataEvaluator.summarize(maml_results),
    ],
    ignore_index=True,
)
summary

## Reproduce from the command line

The same experiment is available as a CLI command:

```bash
themap metalearn datasets/ \
    --distance-file output/molecule_distances.csv \
    --target-id CHEMBL1963831 --k 3 \
    --algorithm proto \
    --support-sizes 16,32,64,128 --seeds 5 \
    --num-epochs 50 -o metalearn_out/proto
```

Swap `--algorithm maml` for the MAML variant. Results (long-form `results.csv`, aggregated
`summary.csv`, selected sources, and training history) are written to the output directory.

**Takeaways to look for:** with sources genuinely close to the target, the meta-learned models
should sit above the from-scratch baseline, with the largest gain at the smallest support sizes —
exactly the low-data regime where transferring knowledge from related datasets matters most.